# 00 · Setup & smoke test
**Goal:** environment works end-to-end on ONE demo before we commit to the full dataset.

Checks: Drive mount → parser install → manifest sync → parse one demo →
round count matches the HLTV scoreboard → positional plot looks like a CS2 round.

> Prerequisite: the `cs2btp` package folder lives at
> `My Drive/cs2-btp/code/cs2btp/` and your demos at `My Drive/cs2-btp/raw_dems/`.

In [ ]:
# --- bootstrap (identical in every notebook) ---------------------------------
from google.colab import drive
drive.mount('/content/drive')
%pip -q install demoparser2 minisom

import sys
sys.path.insert(0, '/content/drive/MyDrive/cs2-btp/code')

from cs2btp import (config as cfg, manifest as mf, parsing, qc,
                    features as ft, roles, consistency as cons,
                    outcome as out, viz)
cfg.ensure_dirs()
print('pipeline ready | drive root =', cfg.DRIVE_ROOT)

## 1 · Sync the manifest with `raw_dems/`
After this cell, open `manifest.csv` in Drive and fill in the **event** and **date** columns — that is your protection against filename collisions.

In [ ]:
mani = mf.sync_with_raw()
mani

## 2 · Parse one demo (smoke test)

In [ ]:
todo = mf.pending('new')
assert len(todo), 'no new demos found in raw_dems/'
row = todo.iloc[0]
print('smoke-testing:', row.filename)
ok = parsing.parse_and_save(row.demo_id, row.filename)
assert ok, 'parse failed - check the traceback above'

## 3 · Validate against HLTV
Compare the score below with the HLTV match page for this demo. They must match exactly (OT included).

In [ ]:
t = parsing.load_parsed(row.demo_id)
rounds = t['rounds']
print(rounds[['round_num','winner_side','round_len_s']].to_string(index=False))
print('\nFinal:', rounds.winner_side.value_counts().to_dict(),
      '| total rounds:', len(rounds))

In [ ]:
rep = qc.check_demo(row.demo_id)
rep

## 4 · Eyeball one round
Orange = T, blue = CT. You should see T players leaving spawn toward a site.

In [ ]:
viz.round_positions(t['ticks'], round_num=2,
                    title=f"{row.demo_id} - round 2");

**If everything above looks right → proceed to `01_parse_all_demos`.**